# LeetCode 205 — Isomorphic Strings bằng Python

**Link đề:** https://leetcode.com/problems/isomorphic-strings/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String

```python
class Solution:
    def isIsomorphic(self, s: str, t: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **ánh xạ một–một** (bijection) giữa các ký tự.
2. Dùng `dict` để lưu ánh xạ, `zip` để duyệt song song hai chuỗi.
3. Thấy vì sao **một dict là chưa đủ** (phản ví dụ) và cần kiểm tra cả hai chiều.
4. Học 4 lời giải: hai dict → dict + set → chuẩn hóa "mẫu chỉ số xuất hiện đầu" → mẹo `len(set(zip))`.
5. Trace từng bước, chứng minh đúng, phân tích Time/Space, tự giải biến thể (Word Pattern).

## 1. Đề bài

Hai chuỗi `s` và `t` gọi là **isomorphic** (đẳng cấu) nếu có thể **thay** các ký tự của `s` để được `t`, với luật:
- Mọi lần xuất hiện của một ký tự phải được thay bằng **cùng** một ký tự.
- Thứ tự giữ nguyên.
- **Hai ký tự khác nhau không được thay thành cùng một ký tự**; một ký tự có thể thay thành chính nó.

| `s` | `t` | Kết quả | Vì sao |
|---|---|:---:|---|
| `"egg"` | `"add"` | `True` | `e→a`, `g→d` |
| `"f11"` | `"b23"` | `False` | `'1'` phải thành cả `'2'` lẫn `'3'` |
| `"paper"` | `"title"` | `True` | `p→t, a→i, e→l, r→e` |
| `"badc"` | `"baba"` | `False` | `b→b` và `d→b`: hai ký tự cùng thành `b` |

**Constraints:** `1 <= len(s) <= 5·10^4`, `len(t) == len(s)`, ký tự ASCII bất kỳ (chữ, số, ký hiệu, khoảng trắng...).

## 2. Diễn đạt bằng ngôn ngữ ánh xạ

Ta tìm một hàm `f` từ ký tự của `s` sang ký tự của `t` sao cho `f(s[i]) == t[i]` với mọi `i`, và:

1. **Nhất quán (hàm):** cùng ký tự `s` → luôn cùng ký tự `t`. Vi phạm: `"f11" / "b23"`.
2. **Đơn ánh (injective):** khác ký tự `s` → khác ký tự `t`. Vi phạm: `"badc" / "baba"`.

Hai điều kiện cùng nhau = **song ánh** (bijection) giữa tập ký tự xuất hiện trong `s` và trong `t`. Khi đó đi ngược `t → s` cũng là một ánh xạ hợp lệ: bài toán **đối xứng**.

## 3. Nền tảng Python

### 3.1. `zip(s, t)` — duyệt song song

`zip` ghép phần tử cùng vị trí của hai dãy thành từng cặp. Với hai chuỗi cùng độ dài, ta nhận các cặp `(s[i], t[i])`.

In [1]:
s, t = "egg", "add"
for a, c in zip(s, t):
    print(a, "<->", c)
print(list(zip("paper", "title")))

e <-> a
g <-> d
g <-> d
[('p', 't'), ('a', 'i'), ('p', 't'), ('e', 'l'), ('r', 'e')]


### 3.2. `enumerate` — vừa lấy chỉ số vừa lấy phần tử

`for i, ch in enumerate(s)` cho `i = 0, 1, 2, ...` cùng ký tự `ch = s[i]`.

In [2]:
for i, ch in enumerate("egg"):
    print("i =", i, "| ch =", ch)

i = 0 | ch = e
i = 1 | ch = g
i = 2 | ch = g


### 3.3. `dict` — bảng tra "khóa → giá trị"

| Thao tác | Cú pháp | Ghi chú |
|---|---|---|
| Tạo rỗng | `m = {}` | |
| Gán | `m['e'] = 'a'` | ghi đè nếu khóa đã có |
| Kiểm tra khóa | `'e' in m` | trung bình `O(1)` |
| Đọc | `m['e']` | lỗi `KeyError` nếu chưa có khóa |
| Đọc an toàn | `m.get('x')` | trả `None` nếu chưa có |
| Đọc có mặc định | `m.setdefault(k, v)` | nếu chưa có thì gán `v` và trả `v`; có rồi thì trả giá trị cũ |

In [3]:
m = {}
m['e'] = 'a'
m['g'] = 'd'
print(m, "| 'e' in m:", 'e' in m, "| m['g'] =", m['g'], "| m.get('x') =", m.get('x'))
print("setdefault('e','z') ->", m.setdefault('e', 'z'), "| setdefault('p','t') ->", m.setdefault('p', 't'))
print(m)

{'e': 'a', 'g': 'd'} | 'e' in m: True | m['g'] = d | m.get('x') = None
setdefault('e','z') -> a | setdefault('p','t') -> t
{'e': 'a', 'g': 'd', 'p': 't'}


### 3.4. `set` — tập hợp không trùng

`used = set()`, `used.add('a')`, `'a' in used`. Ta sẽ dùng set để nhớ "ký tự `t` nào đã có chủ".

In [4]:
used = set()
used.add('a'); used.add('d'); used.add('a')
print(used, "'a' in used?", 'a' in used, "'z' in used?", 'z' in used)

{'a', 'd'} 'a' in used? True 'z' in used? False


## 4. Cách trực giác — một dict `s → t` (và vì sao chưa đủ)

Người mới thường nghĩ: "lưu `s[i] → t[i]`; nếu gặp lại `s[i]` mà nó trỏ tới ký tự khác thì sai".

Cách này kiểm tra điều kiện **(1) nhất quán**, nhưng **quên điều kiện (2) đơn ánh**.

In [5]:
def is_iso_one_dict(s: str, t: str) -> bool:
    m = {}
    for a, c in zip(s, t):
        if a in m and m[a] != c:
            return False
        m[a] = c
    return True

for s, t in [("egg", "add"), ("f11", "b23"), ("paper", "title"), ("badc", "baba")]:
    print(f"{s!r:8} {t!r:8} -> {is_iso_one_dict(s, t)}")

'egg'    'add'    -> True
'f11'    'b23'    -> False
'paper'  'title'  -> True
'badc'   'baba'   -> True


**Phản ví dụ:** `"badc" / "baba"` cho `True` — sai! Dict thu được là `{'b': 'b', 'a': 'a', 'd': 'b', 'c': 'a'}`: cả `b` và `d` đều trỏ tới `'b'`. Một chiều nhất quán, nhưng chiều ngược `t → s` thì `'b'` phải về cả `b` lẫn `d`.

**Nút thắt:** phải đảm bảo không có hai khóa khác nhau cùng một giá trị. Kiểm tra bằng cách quét mọi value trong dict mỗi bước sẽ tốn `O(k)` mỗi lần (`k` = số ký tự khác nhau).

In [6]:
def is_iso_brute(s: str, t: str) -> bool:
    # baseline: kiểm tra đơn ánh bằng cách quét toàn bộ value mỗi lần thêm khóa mới
    m = {}
    for a, c in zip(s, t):
        if a in m:
            if m[a] != c:
                return False
        else:
            if c in m.values():        # quét O(k)
                return False
            m[a] = c
    return True

print(is_iso_brute("badc", "baba"), is_iso_brute("paper", "title"))

False True


## 5. Insight — kiểm tra **cả hai chiều**

Vì bài toán đối xứng, cách gọn nhất là giữ **hai dict**:
- `s_to_t`: ký tự `s` → ký tự `t`.
- `t_to_s`: ký tự `t` → ký tự `s`.

Tại mỗi vị trí `(a, c)`:
- nếu `a` đã có ánh xạ khác `c` → sai (vi phạm nhất quán);
- nếu `c` đã có ánh xạ ngược khác `a` → sai (vi phạm đơn ánh);
- ngược lại ghi nhận cả hai chiều.

Mỗi kiểm tra là một lần tra dict → trung bình `O(1)`.

## 6. Cách 1 — Lời giải chính: hai dict

Từng đoạn:

**Đoạn 1 — độ dài khác thì chắc chắn sai** (đề đảm bảo bằng nhau, nhưng thêm cho an toàn khi dùng ngoài LeetCode).

**Đoạn 2 — duyệt song song**
```python
for a, c in zip(s, t):
```

**Đoạn 3 — kiểm tra hai chiều**
```python
    if s_to_t.get(a, c) != c or t_to_s.get(c, a) != a:
        return False
```
`s_to_t.get(a, c)`: nếu `a` chưa có ánh xạ thì trả **chính `c`** (coi như hợp lệ); nếu có rồi trả ánh xạ cũ để so với `c`. Mẹo này gộp "chưa có" và "có và khớp" vào một phép so sánh.

**Đoạn 4 — ghi nhận**
```python
    s_to_t[a] = c
    t_to_s[c] = a
```

In [7]:
def is_iso_two_dicts(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    s_to_t = {}
    t_to_s = {}
    for a, c in zip(s, t):
        if s_to_t.get(a, c) != c or t_to_s.get(c, a) != a:
            return False
        s_to_t[a] = c
        t_to_s[c] = a
    return True

for s, t in [("egg", "add"), ("f11", "b23"), ("paper", "title"), ("badc", "baba")]:
    print(f"{s!r:8} {t!r:8} -> {is_iso_two_dicts(s, t)}")

'egg'    'add'    -> True
'f11'    'b23'    -> False
'paper'  'title'  -> True
'badc'   'baba'   -> False


## 7. TRACE

### 7.1. Bảng trace `s = "paper"`, `t = "title"`

| i | `a` | `c` | `s_to_t[a]` trước | `t_to_s[c]` trước | Quyết định | `s_to_t` sau |
|---:|:-:|:-:|:-:|:-:|---|---|
| 0 | p | t | — | — | ghi p↔t | {p:t} |
| 1 | a | i | — | — | ghi a↔i | {p:t, a:i} |
| 2 | p | t | t ✓ | p ✓ | khớp | không đổi |
| 3 | e | l | — | — | ghi e↔l | {p:t, a:i, e:l} |
| 4 | r | e | — | — | ghi r↔e | {p:t, a:i, e:l, r:e} |

Kết thúc vòng lặp không có mâu thuẫn → `True`. Lưu ý: `r → e` hợp lệ dù `e` cũng là ký tự của `s` — hai dict tách bạch "ký tự phía `s`" và "ký tự phía `t`".

In [8]:
def trace_iso(s: str, t: str) -> bool:
    s_to_t, t_to_s = {}, {}
    for i, (a, c) in enumerate(zip(s, t)):
        fwd, bwd = s_to_t.get(a), t_to_s.get(c)
        if (fwd is not None and fwd != c) or (bwd is not None and bwd != a):
            why = f"{a!r} đã -> {fwd!r}, không thể -> {c!r}" if fwd is not None and fwd != c else f"{c!r} (bên t) đã là ảnh của {bwd!r}, không nhận thêm {a!r}"
            print(f"i={i}: ({a!r},{c!r}) MÂU THUẪN: {why} -> False")
            return False
        status = "khớp ánh xạ cũ" if fwd == c else "ghi ánh xạ mới"
        s_to_t[a], t_to_s[c] = c, a
        print(f"i={i}: ({a!r},{c!r}) {status:<15} s_to_t={s_to_t}")
    print("Không mâu thuẫn -> True")
    return True

trace_iso("paper", "title")

i=0: ('p','t') ghi ánh xạ mới  s_to_t={'p': 't'}
i=1: ('a','i') ghi ánh xạ mới  s_to_t={'p': 't', 'a': 'i'}
i=2: ('p','t') khớp ánh xạ cũ  s_to_t={'p': 't', 'a': 'i'}
i=3: ('e','l') ghi ánh xạ mới  s_to_t={'p': 't', 'a': 'i', 'e': 'l'}
i=4: ('r','e') ghi ánh xạ mới  s_to_t={'p': 't', 'a': 'i', 'e': 'l', 'r': 'e'}
Không mâu thuẫn -> True


True

### 7.2. Trace nhánh "đơn ánh bị vi phạm" — `"badc"` / `"baba"`

Đây là nhánh mà một dict bỏ sót. Mâu thuẫn phát hiện ở `i = 2` nhờ `t_to_s`.

In [9]:
trace_iso("badc", "baba")
print()
trace_iso("f11", "b23")

i=0: ('b','b') ghi ánh xạ mới  s_to_t={'b': 'b'}
i=1: ('a','a') ghi ánh xạ mới  s_to_t={'b': 'b', 'a': 'a'}
i=2: ('d','b') MÂU THUẪN: 'b' (bên t) đã là ảnh của 'b', không nhận thêm 'd' -> False

i=0: ('f','b') ghi ánh xạ mới  s_to_t={'f': 'b'}
i=1: ('1','2') ghi ánh xạ mới  s_to_t={'f': 'b', '1': '2'}
i=2: ('1','3') MÂU THUẪN: '1' đã -> '2', không thể -> '3' -> False


False

## 8. Cách 2 — một dict + một set "ký tự t đã có chủ"

Thay `t_to_s` bằng `used` (set). Khi gặp ký tự `a` **mới**: nếu `c` đã nằm trong `used` → `c` đã thuộc về một ký tự khác → sai. Chỉ cần biết "đã có chủ hay chưa", không cần biết chủ là ai.

In [10]:
def is_iso_dict_set(s: str, t: str) -> bool:
    mapping = {}
    used = set()
    for a, c in zip(s, t):
        if a in mapping:
            if mapping[a] != c:
                return False
        else:
            if c in used:
                return False
            mapping[a] = c
            used.add(c)
    return True

for s, t in [("egg", "add"), ("f11", "b23"), ("paper", "title"), ("badc", "baba")]:
    print(f"{s!r:8} {t!r:8} -> {is_iso_dict_set(s, t)}")

'egg'    'add'    -> True
'f11'    'b23'    -> False
'paper'  'title'  -> True
'badc'   'baba'   -> False


## 9. Cách 3 — Chuẩn hóa thành "mẫu" (pattern) rồi so sánh

Ý tưởng sáng tạo: thay mỗi ký tự bằng **chỉ số lần xuất hiện đầu tiên** của nó.

```text
"paper" -> p:0 a:1 p:0 e:3 r:4  -> [0, 1, 0, 3, 4]
"title" -> t:0 i:1 t:0 l:3 e:4  -> [0, 1, 0, 3, 4]   giống -> True
"badc"  ->                         [0, 1, 2, 3]
"baba"  ->                         [0, 1, 0, 1]      khác  -> False
```

Hai chuỗi isomorphic ⇔ có cùng "mẫu", vì mẫu chỉ ghi lại **cấu trúc lặp**, bỏ qua ký tự cụ thể. Cách này rất hữu ích khi cần **gom nhóm** nhiều chuỗi (xem biến thể).

In [11]:
def pattern(x: str) -> list[int]:
    first = {}
    out = []
    for i, ch in enumerate(x):
        first.setdefault(ch, i)      # chỉ ghi chỉ số ở lần gặp đầu
        out.append(first[ch])
    return out

for w in ["paper", "title", "badc", "baba", "egg", "add"]:
    print(f"{w:6} -> {pattern(w)}")

def is_iso_pattern(s: str, t: str) -> bool:
    return pattern(s) == pattern(t)

print(is_iso_pattern("paper", "title"), is_iso_pattern("badc", "baba"))

paper  -> [0, 1, 0, 3, 4]
title  -> [0, 1, 0, 3, 4]
badc   -> [0, 1, 2, 3]
baba   -> [0, 1, 0, 1]
egg    -> [0, 1, 1]
add    -> [0, 1, 1]
True False


### 9.1. Biến thể one-liner của cách 3: `map(s.index, s)`

`s.index(ch)` trả vị trí xuất hiện đầu tiên của `ch`. `list(map(s.index, s))` cho đúng mẫu ở trên, nhưng mỗi lần `index` quét từ đầu → `O(n²)` trong trường hợp xấu. Gọn để nhớ, không nên dùng khi `n = 5·10^4` với nhiều ký tự khác nhau.

In [12]:
print(list(map("paper".index, "paper")))
print(list(map("paper".index, "paper")) == list(map("title".index, "title")))

[0, 1, 0, 3, 4]
True


## 10. Cách 4 — Mẹo đếm `len(set(...))`

Quan sát: `s` và `t` isomorphic ⇔ **số ký tự khác nhau** của `s`, của `t`, và **số cặp khác nhau** `(s[i], t[i])` đều bằng nhau.

Lý do trực giác:
- Nếu một ký tự `s` ghép với 2 ký tự `t` khác nhau → số cặp > số ký tự của `s`.
- Nếu hai ký tự `s` cùng ghép với 1 ký tự `t` → số cặp > số ký tự của `t`.
- Mỗi ký tự `s` có ít nhất một cặp, nên số cặp ≥ số ký tự `s`; tương tự với `t`. Bằng nhau ⇔ không có "nhánh" nào ⇔ song ánh.

In [13]:
def is_iso_count(s: str, t: str) -> bool:
    return len(set(s)) == len(set(t)) == len(set(zip(s, t)))

for s, t in [("egg", "add"), ("f11", "b23"), ("paper", "title"), ("badc", "baba")]:
    print(f"{s!r:8} {t!r:8} | set(s)={len(set(s))} set(t)={len(set(t))} cặp={len(set(zip(s, t)))} -> {is_iso_count(s, t)}")

'egg'    'add'    | set(s)=2 set(t)=2 cặp=2 -> True
'f11'    'b23'    | set(s)=2 set(t)=3 cặp=3 -> False
'paper'  'title'  | set(s)=4 set(t)=4 cặp=4 -> True
'badc'   'baba'   | set(s)=4 set(t)=2 cặp=4 -> False


`a == b == c` là **so sánh nối chuỗi** (chained comparison) của Python: tương đương `a == b and b == c`.

## 11. Code nộp LeetCode

In [14]:
class Solution:
    def isIsomorphic(self, s: str, t: str) -> bool:
        s_to_t = {}
        t_to_s = {}
        for a, c in zip(s, t):
            if s_to_t.get(a, c) != c or t_to_s.get(c, a) != a:
                return False
            s_to_t[a] = c
            t_to_s[c] = a
        return True

sol = Solution()
print(sol.isIsomorphic("egg", "add"), sol.isIsomorphic("f11", "b23"), sol.isIsomorphic("paper", "title"))

True False True


Bản ngắn (cách 4) cũng nộp được:

```python
class Solution:
    def isIsomorphic(self, s: str, t: str) -> bool:
        return len(set(s)) == len(set(t)) == len(set(zip(s, t)))
```

## 12. Vì sao cách hai dict đúng?

**Invariant** sau khi xử lý xong `i` vị trí đầu mà chưa trả `False`:

> `s_to_t` và `t_to_s` là hai ánh xạ ngược nhau (song ánh) giữa các ký tự đã gặp của `s[:i]` và `t[:i]`, và `s_to_t[s[j]] == t[j]` với mọi `j < i`.

- **Ban đầu:** hai dict rỗng → đúng.
- **Bước `i`:** nếu `a` đã có ánh xạ khác `c` → không thể tồn tại song ánh cho `s[:i+1]` → trả `False` đúng. Tương tự cho `c` đã có chủ khác `a`. Nếu không mâu thuẫn, thêm cặp `(a, c)` vẫn giữ hai dict là nghịch đảo của nhau.
- **Kết thúc:** duyệt hết, invariant cho `i = n` → tồn tại song ánh biến `s` thành `t` → `True` đúng.

Ngược lại, mỗi lần trả `False` đều có một "nhân chứng" cụ thể (một ký tự hai đích, hoặc hai ký tự một đích) → không trả nhầm.

## 13. Time / Space — so sánh

Gọi `n = len(s)`, `k` = số ký tự khác nhau (với ASCII, `k <= 128` — hằng số).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Một dict (sai) | `O(n)` | `O(k)` | bỏ sót đơn ánh |
| Baseline `c in m.values()` | `O(n·k)` | `O(k)` | đúng nhưng chậm hơn |
| 1. Hai dict | `O(n)` | `O(k)` | **chuẩn** |
| 2. Dict + set | `O(n)` | `O(k)` | tương đương |
| 3. Mẫu chỉ số đầu | `O(n)` | `O(n)` (list mẫu) | tốt cho gom nhóm |
| 3'. `map(s.index, s)` | `O(n·k)` | `O(n)` | gọn, dễ nhớ |
| 4. `len(set(zip))` | `O(n)` | `O(min(n, k²))` cho set các cặp | ngắn nhất |

Tra/ghi dict, set đều là trung bình `O(1)` nhờ hash.

## 14. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("ví dụ 1 egg/add", sol.isIsomorphic("egg", "add"), True)
check("ví dụ 2 f11/b23", sol.isIsomorphic("f11", "b23"), False)
check("ví dụ 3 paper/title", sol.isIsomorphic("paper", "title"), True)
check("foo/bar (mẫu đề)", sol.isIsomorphic("foo", "bar"), False)
check("phản ví dụ một dict badc/baba", sol.isIsomorphic("badc", "baba"), False)
check("1 ký tự", sol.isIsomorphic("a", "z"), True)
check("ánh xạ chính nó", sol.isIsomorphic("abc", "abc"), True)
check("hoán vị chéo ab/ba", sol.isIsomorphic("ab", "ba"), True)
check("khoảng trắng & ký hiệu", sol.isIsomorphic("a b!", "x y?"), True)
check("ký tự trùng phía t: ab/aa", sol.isIsomorphic("ab", "aa"), False)
check("ký tự trùng phía s: aa/ab", sol.isIsomorphic("aa", "ab"), False)

PASS ví dụ 1 egg/add: True
PASS ví dụ 2 f11/b23: False
PASS ví dụ 3 paper/title: True
PASS foo/bar (mẫu đề): False
PASS phản ví dụ một dict badc/baba: False
PASS 1 ký tự: True
PASS ánh xạ chính nó: True
PASS hoán vị chéo ab/ba: True
PASS khoảng trắng & ký hiệu: True
PASS ký tự trùng phía t: ab/aa: False
PASS ký tự trùng phía s: aa/ab: False


In [16]:
import random
rng = random.Random(205)

def oracle(s, t):
    # định nghĩa trực tiếp: tồn tại song ánh <=> map 2 chiều nhất quán, kiểm tra bằng brute force
    return all((s[i] == s[j]) == (t[i] == t[j]) for i in range(len(s)) for j in range(len(s)))

fns = [is_iso_brute, is_iso_two_dicts, is_iso_dict_set, is_iso_pattern, is_iso_count, Solution().isIsomorphic]
for _ in range(4000):
    n = rng.randint(1, 8)
    s = "".join(rng.choice("abc") for _ in range(n))
    t = "".join(rng.choice("xyz") for _ in range(n))
    exp = oracle(s, t)
    for f in fns:
        assert f(s, t) == exp, (f.__name__, s, t)
print("6 cài đặt khớp oracle trên 4000 cặp ngẫu nhiên")
print("Một dict sai ở bao nhiêu ca?", sum(is_iso_one_dict(s, t) != oracle(s, t)
      for s, t in [("".join(rng.choice("abc") for _ in range(5)), "".join(rng.choice("xy") for _ in range(5))) for _ in range(500)]))

6 cài đặt khớp oracle trên 4000 cặp ngẫu nhiên
Một dict sai ở bao nhiêu ca? 104


Oracle ở trên dùng một định nghĩa tương đương rất trực tiếp: với mọi cặp vị trí `i, j`, "`s[i] == s[j]`" phải tương đương "`t[i] == t[j]`". Chậm `O(n²)` nhưng hiển nhiên đúng → dùng để kiểm tra các cách nhanh.

## 15. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra một chiều
Đã thấy ở mục 4: `"badc" / "baba"` trả `True` sai. Sửa: thêm `t_to_s` hoặc `used`.

### Lỗi 2 — Dùng `m[a]` khi `a` chưa có → `KeyError`

In [17]:
m = {}
try:
    if m['x'] != 'y':
        pass
except KeyError as e:
    print("KeyError:", e, "-> dùng 'x' in m hoặc m.get('x')")

KeyError: 'x' -> dùng 'x' in m hoặc m.get('x')


### Lỗi 3 — So sánh **tần suất** thay vì **cấu trúc**

```python
sorted(Counter(s).values()) == sorted(Counter(t).values())   # SAI
```
`"aab"` và `"aba"` có cùng tần suất `{2, 1}` nhưng **không** isomorphic (`aab` → mẫu `[0,0,2]`, `aba` → `[0,1,0]`). Tần suất không giữ **thứ tự**.

In [18]:
from collections import Counter
s, t = "aab", "aba"
print("Tần suất giống?", sorted(Counter(s).values()) == sorted(Counter(t).values()))
print("Isomorphic thật?", Solution().isIsomorphic(s, t))

Tần suất giống? True
Isomorphic thật? False


### Lỗi 4 — Dùng một dict chung cho cả hai chiều

```python
m = {}
m[a] = c
m[c] = a      # SAI: ký tự 'e' trong s và 'e' trong t bị lẫn vào nhau
```
Với `"paper"/"title"`: có `r → e` và `e → l` — nếu dùng chung dict, `m['e']` bị ghi đè lẫn lộn. Phải tách hai dict.

In [19]:
def is_iso_bug_shared(s, t):
    m = {}
    for a, c in zip(s, t):
        if (a in m and m[a] != c) or (c in m and m[c] != a):
            return False
        m[a] = c
        m[c] = a
    return True

print("Bản dict chung, paper/title:", is_iso_bug_shared("paper", "title"), "(đúng phải là True)")

Bản dict chung, paper/title: False (đúng phải là True)


## 16. Mẹo nhận dạng pattern & biến thể

**Dấu hiệu:** "thay thế ký tự", "cùng mẫu", "ánh xạ một–một", "bijection", "pattern matching" → dict hai chiều hoặc chuẩn hóa mẫu.

Bài liên quan:
- **290 Word Pattern:** `pattern = "abba"`, `s = "dog cat cat dog"` — giống hệt nhưng phía `t` là **từ** (dùng `s.split()`).
- **890 Find and Replace Pattern:** lọc các từ có cùng mẫu với `pattern` → dùng hàm `pattern()`.
- **242 Valid Anagram:** so tần suất (khác bài này — không quan tâm thứ tự).

### Biến thể 1 — Word Pattern
### Biến thể 2 — Gom nhóm các từ cùng mẫu

In [20]:
def word_pattern(pat: str, s: str) -> bool:
    words = s.split()                     # tách theo khoảng trắng
    if len(pat) != len(words):
        return False
    p2w, w2p = {}, {}
    for ch, w in zip(pat, words):
        if p2w.get(ch, w) != w or w2p.get(w, ch) != ch:
            return False
        p2w[ch], w2p[w] = w, ch
    return True

print(word_pattern("abba", "dog cat cat dog"), word_pattern("abba", "dog dog dog dog"))

def group_isomorphic(words: list[str]) -> list[list[str]]:
    groups = {}
    for w in words:
        key = tuple(pattern(w))           # list không làm khóa dict được -> đổi sang tuple
        groups.setdefault(key, []).append(w)
    return list(groups.values())

print(group_isomorphic(["egg", "add", "foo", "abc", "xyz", "paper", "title", "bar"]))

True False
[['egg', 'add', 'foo'], ['abc', 'xyz', 'bar'], ['paper', 'title']]


`tuple(...)`: list là **mutable** (thay đổi được) nên không băm được, không làm khóa dict được; tuple là **immutable** nên làm khóa được.

## 17. Bài tập tự luyện

### Bài 1 — Tính tay
Viết "mẫu chỉ số xuất hiện đầu" cho `"abcabc"`, `"xyzxyz"`, `"aabb"`. Cặp nào isomorphic?

### Bài 2 — Dự đoán
`is_iso_one_dict("ab", "aa")` trả gì? Đúng hay sai so với đề?

### Bài 3 — TODO: viết cách hai dict

In [21]:
def exercise_two_dicts(s: str, t: str) -> bool:
    # TODO 1: tạo s_to_t, t_to_s
    # TODO 2: duyệt zip(s, t), kiểm tra hai chiều
    # TODO 3: ghi nhận, cuối cùng trả True
    pass

# assert exercise_two_dicts("egg", "add") is True
# assert exercise_two_dicts("badc", "baba") is False

### Bài 4 — TODO: viết hàm `pattern` và cách 3

In [22]:
def exercise_pattern(x: str) -> list[int]:
    # TODO: mỗi ký tự -> chỉ số lần xuất hiện đầu
    pass

# assert exercise_pattern("paper") == [0, 1, 0, 3, 4]

### Bài 5 — Biến thể: Word Pattern (LeetCode 290)

In [23]:
def exercise_word_pattern(pat: str, s: str) -> bool:
    # TODO: tách s thành từ, kiểm tra độ dài, ánh xạ hai chiều ký tự <-> từ
    pass

# assert exercise_word_pattern("abba", "dog cat cat dog") is True
# assert exercise_word_pattern("abba", "dog cat cat fish") is False
# assert exercise_word_pattern("aaa", "dog dog") is False

### Bài 6 — Mini challenge: Find and Replace Pattern (LeetCode 890)
Cho `words` và `pat`, trả về các từ isomorphic với `pat` (giữ thứ tự).
`find_and_replace(["abc","deq","mee","aqq","dkd","ccc"], "abb")` → `["mee","aqq"]`.

In [24]:
def exercise_find_and_replace(words: list[str], pat: str) -> list[str]:
    # TODO
    pass

# assert exercise_find_and_replace(["abc","deq","mee","aqq","dkd","ccc"], "abb") == ["mee", "aqq"]

## Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Ghi chỉ số khi gặp ký tự lần đầu | dùng dict `first` | `abcabc → [0,1,2,0,1,2]` |
| 2 | Một dict có kiểm tra đơn ánh không? | `{'a':'a','b':'a'}` | trả `True` — sai |
| 3 | Hai chiều | `s_to_t.get(a, c) != c` | `or t_to_s.get(c, a) != a` |
| 4 | `setdefault` | `first.setdefault(ch, i)` | `out.append(first[ch])` |
| 5 | `t` là list từ | `words = s.split()` | kiểm tra `len(pat) != len(words)` trước |
| 6 | Tái dùng hàm mẫu | `pattern(w) == pattern(pat)` | list comprehension `[w for w in words if ...]` |

## Đáp án tham khảo

**Bài 1:** `"abcabc" → [0,1,2,0,1,2]`, `"xyzxyz" → [0,1,2,0,1,2]`, `"aabb" → [0,0,2,2]`. Chỉ `abcabc` và `xyzxyz` isomorphic.

**Bài 2:** trả `True` — **sai**, vì `a` và `b` cùng thành `a`. Đề yêu cầu `False`.

In [25]:
def answer_two_dicts(s, t):
    s_to_t, t_to_s = {}, {}
    for a, c in zip(s, t):
        if s_to_t.get(a, c) != c or t_to_s.get(c, a) != a:
            return False
        s_to_t[a] = c
        t_to_s[c] = a
    return True

def answer_pattern(x):
    first, out = {}, []
    for i, ch in enumerate(x):
        first.setdefault(ch, i)
        out.append(first[ch])
    return out

def answer_word_pattern(pat, s):
    words = s.split()
    if len(pat) != len(words):
        return False
    return answer_two_dicts(pat, words)     # zip hoạt động với list từ y như chuỗi

def answer_find_and_replace(words, pat):
    target = answer_pattern(pat)
    return [w for w in words if len(w) == len(pat) and answer_pattern(w) == target]

assert answer_pattern("abcabc") == answer_pattern("xyzxyz") == [0, 1, 2, 0, 1, 2]
assert answer_pattern("aabb") == [0, 0, 2, 2]
assert is_iso_one_dict("ab", "aa") is True and Solution().isIsomorphic("ab", "aa") is False
assert answer_two_dicts("egg", "add") is True and answer_two_dicts("badc", "baba") is False
assert answer_pattern("paper") == [0, 1, 0, 3, 4]
assert answer_word_pattern("abba", "dog cat cat dog") is True
assert answer_word_pattern("abba", "dog cat cat fish") is False
assert answer_word_pattern("aaa", "dog dog") is False
assert answer_find_and_replace(["abc", "deq", "mee", "aqq", "dkd", "ccc"], "abb") == ["mee", "aqq"]
print("Đáp án tham khảo PASS toàn bộ!")

Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
fwd, bwd = {}, {}
for a, c in zip(s, t):
    if fwd.get(a, c) != c or bwd.get(c, a) != a:
        return False          # một ký tự hai đích, hoặc hai ký tự một đích
    fwd[a], bwd[c] = c, a
return True
```

Thay thế nhanh: `len(set(s)) == len(set(t)) == len(set(zip(s, t)))`.

**Checklist:**
- [ ] Kiểm tra **cả hai chiều** (thử `"badc"/"baba"`, `"ab"/"aa"`).
- [ ] Không dùng chung một dict cho hai phía.
- [ ] Không so tần suất thay cho cấu trúc.
- [ ] Không đọc `m[key]` khi chưa chắc có khóa.
- [ ] Tên method `isIsomorphic`, trả `bool`.